In [188]:
import pandas as pd
from Levenshtein import distance
import numpy as np
import re
import unicodedata

In [189]:
#read the Compensation Office csv file into a dataframe
df_csv = pd.read_excel('/home/Benha/Projects/Arbeit/FIZ/compensation-office-analysis/data_new/2025_06_03_Liste der Entschädigungsämter mit Adressen, Varianten, Normalisierungen und Archiven.xlsx')
df_csv.sample(20)

,Layout class,Filename,Bundesland,Transkription Compensation Office,Normalisierung Compensation Office1,Normalisierung Compensation Office2,Zuständiges Archiv,Zuständiges Archiv - GND-Nummer,Unnamed: 8
59,BY-Spätphase (abweichend 1),1922_02_18_25_0,Bayern,Bayerisches \nLandesentschädigungsamt\nPrinz-L...,Bayerisches Landesentschädigungsamt München,NaN,Bayerisches Hauptstaatsarchiv,2005486-5,NaN
390,RLP-Frühe-Phase,1903_01_16_53_0,Rheinland-Pfalz,Regierungsbezirksamt\nfür Wiedergutmachung\nun...,Regierungsbezirksamt für Wiedergutmachung und ...,NaN,Landesarchivverwaltung Rheinland-Pfalz,2029741-5,NaN
382,RLP-Frühe-Phase,1903_01_18_77_0,Rheinland-Pfalz,Regierungsbezirksamt\nfür Wiedergutmachung\nun...,Regierungsbezirksamt für Wiedergutmachung und ...,NaN,Landesarchivverwaltung Rheinland-Pfalz,2029741-5,NaN
133,Hinweiskarte_schwarzes_Datum,1903_03_20_53_0,Schleswig-Holstein,Kiel,Kiel,NaN,Landesarchiv Schleswig-Holstein,2017048-8,NaN
106,HH-NI-NRW-SH-Hauptphase,1897_05_15_19_0,Nordrhein-Westfalen,b) Der Oberkreisdirektor -Amt für Wiedergutmac...,"Der Oberkreisdirektor, Amt für Wiedergutmachun...",NaN,Landesarchiv Nordrhein-Westfalen,10094504-1,NaN
257,offizielle Bezeichnungen und Adressen,NaN,Nordrhein-Westfalen,Der Regierungspräsident in Arnsberg,NaN,NaN,Landesarchiv Nordrhein-Westfalen,10094504-1,NaN
80,HH-NI-NRW-SH-Hauptphase,1880_03_30_6_0,Schleswig-Holstein,Landesentschädigungsamt\nSchleswig-Holstein in...,Landesentschädigungsamt Schleswig-Holstein in ...,NaN,Landesarchiv Schleswig-Holstein,2017048-8,NaN
48,BY-HB-Frühe Phase,1850_03_13_1_0,Baden-Württemberg,Karlsruhe,Karlsruhe,NaN,Landesarchiv Baden-Württemberg,10107080-9,NaN
112,HH-NI-NRW-SH-Hauptphase,1873_02_09_4_0,Hamburg,"Hansestadt Hamburg, Sozialbehörde, Amt für Wie...","Sozialbehörde, Amt für Wiedergutmachung Hamburg",NaN,Staatsarchiv (Hamburg),2022556-8,NaN
243,offizielle Bezeichnungen und Adressen,NaN,Rheinland-Pfalz,Wiedergutmachungsamt Rheinland-Pfalz Berlin\nB...,NaN,NaN,Amt für Wiedergutmachung Saarburg,1354917804,NaN


In [190]:
#creaae a list of unique compensation offices from the csv file
gt = df_csv['Transkription Compensation Office '].unique().tolist()
print("Number of unique compensation offices: ", len(gt))

Number of unique compensation offices:  428


In [191]:
#read the sliced extracted data jsonl file into a dataframe
df_json = pd.read_json('compoff_extracted.jsonl', lines=True)
df_json.sample(20)

,CompensationOffice1,BZKNr,filename
348454,Köln,671 648,1934_01_07_15_0.jpg
109026,Hessen,D/7250/03/A/Rot,1903_07_14_84_0.jpg
164938,Niedersachsen,120986,1922_04_12_81_0.jpg
929114,Köln,710759,1936_04_25_6_0.jpg
1050986,NaN,66.481/III/7151,1940_04_30_9_0.jpg
1036680,Braunschweig,8-02385,1895_05_10_5_0.jpg
546249,Entschädigungsbehörde Köln,Art. V-56-II-Nr. 18 536,1915_04_08_37_0.jpg
279830,Hessen,I 6 W / 35399,1921_10_27_45_0.jpg
1115863,Bezirksamt für Wiedergutmachung KOBLENZ,120488,1899_12_27_16_0.jpg
650864,Freie und Hansestadt Hamburg Sozialbehörde Amt...,220405,1905_04_22_51_0.jpg


In [192]:
#create a dataframe with the entries that have a non-null value in the CompensationOffice1 column
print("Number of cards: ",len(df_json))
mask_not_null = df_json['CompensationOffice1'].notna()
print(f'Number of non-null entries: {mask_not_null.sum()}')


Number of cards:  1901284
Number of non-null entries: 1825946


In [193]:
#create a function to normalise the compensation office names by removing punctuation and whitespace, converting to lowercase and replacing umlauts with their equivalent

#def normalise(word):
#    return re.sub(r'\s+', ' ', str(word)).strip().lower()
def normalize(s):
    if not isinstance(s, str):
        return ""
    
    s = s.lower()

    # Normalize umlauts and ß
    replacements = {
        "ä": "ae", "ö": "oe", "ü": "ue", "ß": "ss"
    }
    for k, v in replacements.items():
        s = s.replace(k, v)
    
    # Normalize unicode (just in case)
    s = unicodedata.normalize("NFKD", s)
    
    # Remove punctuation and whitespace
    s = re.sub(r'[\W_]+', '', s)
        
    return s

gt_normalized = [normalize(office) for office in gt]

In [194]:
#create a mask for the entries in the json dataframe that match any of the normalized compensation office names in the gt list
mask_matched_0 = df_json['CompensationOffice1'].apply(normalize).isin(gt_normalized)
print(f'Number of matches: {mask_matched_0.sum()}')

df_unmatched = df_json[mask_not_null & ~mask_matched_0]
print(f'Number of unmatched entries: {len(df_unmatched)}')


Number of matches: 1494694
Number of unmatched entries: 331252


In [195]:
#print the unique values in the CompensationOffice1 column of the unmatched dataframe and their counts
print(len(df_unmatched['CompensationOffice1'].unique().tolist()))
df_unmatched['CompensationOffice1'].value_counts()

27629


CompensationOffice1
Bezirksamt für Wiedergutmachung Köln                                   32785
Hess. Staatsministerium                                                29978
Freie und Hansestadt Hamburg Sozialbehörde Amt für Wiedergutmachung    16429
Landesamt für die Wiedergutmachung                                     14641
Hansestadt Hamburg, Sozialbehörde, Amt für Wiedergutmachung            11836
                                                                       ...  
Regierung Aachen (Welt.)                                                   1
Regierungspräsident Arasberg                                               1
Kreisssonderhilfsausschuss des Landkreises Garsch. Schlaumburg             1
Bayerisches Landesamt für den Regierungsbezirk München                     1
Reg.-Präsident /Präsident des Vereinsbezirks Hannover                      1
Name: count, Length: 27629, dtype: int64

In [196]:
#create a list of manually corrected compensation office names that are not in the gt list but are in the json dataframe and normalise them
list_corrections = ["Bezirksamt für Wiedergutmachung Köln", "Hess. Staatsministerium", "Freie und Hansestadt Hamburg Sozialbehörde Amt für Wiedergutmachung", "Hansestadt Hamburg, Sozialbehörde, Amt für Wiedergutmachung", 
                    "Der Innenminister des Landes Nordrhein-Westfalen", "Landesamt für die Wiedergutmachung Baden-Württemberg", "Oberfinanzdirektion Niedersachsen", "Statistisches Landesamt Nordrhein-Westfalen" , "Oberfinanzdirektion Niedersachsen"
                    , "Rhein. Pfalz - Berlin", "Regierungsbezirksamt für Wiedergutmachung und verwaltete Vermögen Köln", "Aachen", "Landesbeirat für die Wiedergutmachung, Stuttgart", "Freiburg i.Br.", "Hamburg, Sozialbehörde, Amt für Wiedergutmachung",
                    "Freie und Hansestadt Hamburg", "Regierungsbezirk Mainz", "Hessisches Ministerium für Jugend, Familie und Gesundheit" , "Reg.-Präsident / Präsident des Verw.-Bezirks Hannover", "Amt für Wiedergutmachung des Landes Rheinland/Pfalz in Berlin Hauptabteilung II",
                    "Schleswig-Holstein in Kiel", "Kreissozialamt Hannover", "Reg.-Präsident Hannover", "Amt für Wiedergutmachung Rheinland-Pfalz" , "Saarburg Bezirksamt für Wiedergutmachung", "Freie und Hansestadt Hamburg, Arbeits- und Sozialbehörde, Amt für Wiedergutmachung",
                    "Niedersachsen-Präsident / Präsident des Verwaltungsbezirks Hannover", "Regierungsbezirksamt für Wiedergutmachung und verwaltete Vermögen Neustadt a. d. Weinstraße", "KSHA Hannover", "Regierungsbezirk für Wiedergutmachung und verwaltete Vermögen Trier",
                    "Oberfinanzdirektion Magdeburg", "Regierungsbezirkskammer für Wiedergutmachung und verwaltete Vermögen Mainz","Regierungsamt Arnsberg (Westf.)", "Regierungsbezirkskammer für Wiedergutmachung und verwaltete Vermögen Mainz",
                    "Regierungsbezirksamt für Wiedergutmachung und verwaltete Vermögen München", "Landesamt für die Wiedergutmachung Baden-Württemberg Stuttgart"]

list_corrections_normalized = [normalize(office) for office in list_corrections]
gt_completed = gt_normalized + list_corrections_normalized
df_json['CompensationOffice1'] = df_json['CompensationOffice1'].str.replace("Bayrische", "Bayerische")
mask_matched_1 = df_json['CompensationOffice1'].apply(normalize).isin(gt_completed)
print(f'Number of matches: {mask_matched_1.sum()}')
df_unmatched = df_json[mask_not_null & ~mask_matched_1]
print(f'Number of unmatched entries: {len(df_unmatched)}')
print(f"Unique unmatched entries: {len(df_unmatched['CompensationOffice1'].unique().tolist())}")
percentage = len(df_unmatched) / len(mask_not_null) * 100
print(f"Percentage of unmatched entries: {percentage:.2f}%")



Number of matches: 1628175
Number of unmatched entries: 197771
Unique unmatched entries: 27325
Percentage of unmatched entries: 10.40%


In [197]:
df_unmatched['CompensationOffice1'].value_counts().head(30)

CompensationOffice1
Landesamt für die Wiedergutmachung                                       14641
Bayerische Kartei                                                         9925
Regierungsbezirksamt für Wiedergutmachung und verwaltete Vermögen         7261
Bundeskartei                                                              4884
Bezirksamt für Wiedergutmachung                                           4130
HNG                                                                       3402
Regierungsbezirksamt für Wiedergutmachung und kontrollierte Vermögen      3354
Bayerisches Landesarchiv                                                  2585
Regierungsbezirks-Amt für Wiedergutmachung und kontrollierte Vermögen     2557
Bezirksamt für Wiedergutmachung 65 Mainz                                  1304
Bayerisches Landesamt für Verfolgtenfragen                                1291
Kreissonderhilfsausschuss                                                 1101
Barburg                         

# Matching based on Levenshtein Distance

In [198]:
#  return the minimum distance and the best match up to a threshold of half the length of the cleaned name
def get_min_distance_dynamic_normalized(name):
    name_clean = normalize(name)
    threshold = len(name_clean) // 2
    
    min_dist = float('inf')
    best_match = None
    for office in gt_completed:
        office_clean = normalize(office)
        d = distance(name_clean, office_clean)
        if d == 0:
            return 0, office
        if d < min_dist:
            min_dist = d
            best_match = office
    if min_dist <= threshold:
        return min_dist, best_match
    return np.nan, None

results = df_unmatched['CompensationOffice1'].apply(get_min_distance_dynamic_normalized)

lev_matches = pd.DataFrame({
    'CompensationOffice1': df_unmatched['CompensationOffice1'],
    'MatchedOffice': results.apply(lambda x: x[1]),
    'MatchDistance': results.apply(lambda x: x[0])
})


In [199]:
#print the matches with a distance of 1 and 2
df_matches_2 = lev_matches[lev_matches['MatchDistance'].isin([1, 2])]
df_matches_2.sample(25)

,CompensationOffice1,MatchedOffice,MatchDistance
61649,Landesentschädigungsamt,blandesentschaedigungsamt,1.0
847978,Bezirksamt für Wiedergutmachung 65 Mainz,bezirksamtfuerwiedergutmachungmainz,2.0
1284021,Bezirksamt für Wiedergutmachung NEUSTADT c. d....,bezirksamtfuerwiedergutmachungneustadtadweinstr,1.0
1390786,Landesamt für die Wiedergutmachung Stuttgart-S.,landesamtfuerdiewiedergutmachungstuttgart,1.0
1639743,Regierungsbezirksamt für Wiedergutmachung und ...,regierungsbezirksamtfuerwiedergutmachungundver...,1.0
1684729,Regierungsamt Arnberg (Westf.),regierungsamtarnsbergwestf,1.0
1345837,Bezirksamt für Wiedergutmachung 55 TRIER,bezirksamtfuerwiedergutmachungtrier,2.0
1617700,Kön,koeln,1.0
1226362,Bezirksamt für Wiedergutmachung Koblinz,bezirksamtfuerwiedergutmachungkoblenz,1.0
363102,Sarbrüken,saarbruecken,2.0


In [200]:
#print the counts of the match distances of 1 and 2
df_matches_2['MatchDistance'].value_counts()
print(f"Number of matches with distance 1 or 2: {len(df_matches_2)}")
print(f"Number of unmatched entries after adding the close matches: {len(df_unmatched) - len(df_matches_2)}")

Number of matches with distance 1 or 2: 27826
Number of unmatched entries after adding the close matches: 169945


In [201]:
#remove the matches with distance 1 and 2 from the unmatched dataframe
df_unmatched_filtered = df_unmatched[~df_unmatched['CompensationOffice1'].isin(lev_matches[lev_matches['MatchDistance'].isin([1, 2])]['CompensationOffice1'])]
df_unmatched_filtered['CompensationOffice1'].value_counts().iloc[30:40]

CompensationOffice1
Amt für Wiedergutmachung des Landes Rheinland-Pfalz                470
Regierung Arnsberg (Wf.)                                           459
Doppelmeldung                                                      449
Amt für Wiedergutmachung Stadt Bonn                                447
Bundesamt für Wiedergutmachung                                     439
Niedersachsen-Präsident / Präsident des Verwaltungsgerichtshofs    437
Regierung Köln                                                     421
Bundesarchiv                                                       408
Reg.-Präsidium Hildesheim                                          396
Entscheidungsbehörde Köln                                          396
Name: count, dtype: int64

In [202]:
#print the top 200 unmatched entries
v_sum = 0

for n, v in df_unmatched_filtered['CompensationOffice1'].value_counts().iloc[:200].to_dict().items():
    print(f"{n}: {v}")
    v_sum += v

print(f"Total count of the top 200 unmatched entries: {v_sum}")

Landesamt für die Wiedergutmachung: 14641
Bayerische Kartei: 9925
Regierungsbezirksamt für Wiedergutmachung und verwaltete Vermögen: 7261
Bundeskartei: 4884
Bezirksamt für Wiedergutmachung: 4130
HNG: 3402
Regierungsbezirksamt für Wiedergutmachung und kontrollierte Vermögen: 3354
Bayerisches Landesarchiv: 2585
Regierungsbezirks-Amt für Wiedergutmachung und kontrollierte Vermögen: 2557
Bayerisches Landesamt für Verfolgtenfragen: 1291
Kreissonderhilfsausschuss: 1101
Kreissozialamt: 958
Bundesminister der Finanzen: 904
Entschädigungsbehörde: 863
Bundesamt: 834
König: 825
Bayerisches Landesamt für Vertriebene: 815
Reg.-Präsident / Präsident des Verw.-Bezirks: 752
KSHA: 750
Der Herr Regierungspräsident in Detmold: 745
Kreissozialhilfsausschuss: 712
Landesamt für Wiedergutmachung: 586
Bundesamt für Vertriebene: 560
Landesamt für Vertriebene: 541
Bayerisches Landesamt für Verfolgtenentschädigung: 521
Landesentschädigungsamt München 2, Arcisstraße 11: 501
Oberfinanzdirektion Niedersachsen Lande

In [203]:
list_further_corrections = ['Oberfinanzdirektion Rostock', 'Niedersächsisches Landesamt für Bezüge und Versorgung - Wiedergutmachung',
                        'Regierungspräsidium Arnsberg', 'Reg.-Präsident / Präsident des Verwaltungsbezirks Hannover', 
                        'Regierungsbezirksamt für Wiedergutmachung und verwaltete Vermögen Neustadt/w.', 'Montabaur', 
                        'Der Regierungspräsidium Düsseldorf', 'Magistrat der Stadt Bremen', 'Freie Hansestadt Bremen', 
                        'Freie Hansestadt Bremen - Landesamt für Wiedergutmachung', 'Reg.-Präsident / Präsident des Verw.-Bezirks Oldenburg', 
                        'Landesentschädigungsamt München', 'Bayerisches Hauptstaatsarchiv', 'Reg.-Präsident / Präsident des Verw.-Bezirks Braunschweig', 
                        'Niedersächsisches Landesverwaltungsamt - Wiedergutmachung', 'Bayerisches Landesamt für Verfolgungsgeschädigte', 
                        'Regierung Arnsberg', 'Regierungsbezirksamt für Wiedergutmachung und verwaltete Vermögen Neustadt/W.', 'Erfurt', 
                        'Entscheidungsbehörde Köln', 'Reg.-Präsidium Hildesheim', 'Bundesarchiv', 'Regierung Köln', 
                        'Niedersachsen-Präsident / Präsident des Verwaltungsgerichtshofs', 'Amt für Wiedergutmachung Stadt Bonn', 
                        'Regierung Arnsberg (Wf.)', 'Amt für Wiedergutmachung des Landes Rheinland-Pfalz', 
                        'Köln, Riehler Pl.2, 5000 Köln 1', 'Oberfinanzdirektion Niedersachsen Landesweite Bezüge- und Versorgungsstelle - Wiedergutmachung -', 
                        'Landesentschädigungsamt München 2, Arcisstraße 11', 'Bayerisches Landesamt für Verfolgtenentschädigung', 
                        'Bundesamt für Vertriebene', 'Der Herr Regierungspräsident in Detmold', 'Bayerisches Landesamt für Vertriebene', 
                        'Bayerisches Landesarchiv', 'Bayerisches Landesamt für Verfolgtenfragen', 'Bundesminister der Finanzen',
                        'Niedersachsen-Präsident / Präsident des Verw.-Bezirks', 'Frankfurt/M.', 'Regierungsamt Aachen (Westf.)', 
                        'Regierungspräsidium Münster', 'Ansbach', 'Niedersachsen-Präsident / Präsident des Verwaltungsbezirks Lüneburg',
                        'Regierung Ansbach (Würt.)', 'Regierungsbezirk für Wiedergutmachung und verwaltete Vermögen Neustadt a.d. Weinstr.',
                        'Aurich/Ostfriesland', 'Regionalverwaltungsbehörde für Wiedergutmachung und verwaltete Vermögen Koblenz',
                        'Entschädigungsbüro Köln', 'Magistrat der Stadt Bremenhaven', 'Amt für Wiedergutmachung des Landes Rheinland/Pfalz',
                        'Kreissonderhilfsausschuss Hannover', 'Bayerisches Verfolgtenamt', 'KSHA Hannover-Land', 'Regierungsbezirk Köln',
                        'Köln, Riehler Pl. 2, 5000 Köln 1', 'Regierungsbezirk Arnsberg (Westf.)', 'Oberfinanzdirektion Erfurt', 
                        'Regierungsbezirk Trier', '']

In [204]:
list_no_sufficient_information = ['Niedersachsen-Präsident / Präsident des Verwaltungsbezirks', 'An Bundeskartei', 
                              'Bundesamt für Wiedergutmachung', 'Amt für Wiedergutmachung und kontrollierte Vermögen', 
                              'Oberfinanzdirektion: Bundesvermögensabteilung', 'Landesamt für Vertriebene', 
                              'Landesamt für Wiedergutmachung', 'Kreissozialhilfsausschuss', 'Reg.-Präsident / Präsident des Verw.-Bezirks', 
                              'Entschädigungsbehörde', 'Kreissozialamt', 'Kreissonderhilfsausschuss', 'Regierungsbezirks-Amt für Wiedergutmachung und kontrollierte Vermögen', 
                              'Regierungsbezirksamt für Wiedergutmachung und kontrollierte Vermögen', 'Bundeskartei', 
                              'Regierungsbezirksamt für Wiedergutmachung und verwaltete Vermögen', 'Kreissozialamt', 'Bundesamt', 'Doppelmeldung', 
                              "Landesamt für die Wiedergutmachung", "Bezirksamt für Wiedergutmachung", "Kreissozialamt", "König",
                              'Regierungsbezirksamt für Wiedergutmachung und kontrolliertes Vermögen', 'Sondervermögens-u. Bauverwaltung',
                              'Bayerisches Landesamt', 'Bundesfinanzdirektion West', 'KSHA', 'KSH', 'Regierungsbezirk für Wiedergutmachung und kontrollierte Vermögen',
                              ]

In [205]:
print(len(list_no_sufficient_information))

30


In [206]:
#filtering out impossible matches and further additions to gt
for office in list_no_sufficient_information + list_further_corrections:
    if office in df_unmatched_filtered['CompensationOffice1'].values:
        df_unmatched_filtered = df_unmatched_filtered[df_unmatched_filtered['CompensationOffice1'] != office]

print(len(df_unmatched_filtered))
print(len(df_unmatched_filtered['CompensationOffice1'].unique()))

98173
24451


In [207]:
df_matches_correctable = lev_matches.dropna()
print(len(df_matches_correctable))
df_matches_impossible = lev_matches[lev_matches['MatchedOffice'].isna()]
impossible_list = df_matches_impossible['CompensationOffice1'].tolist()
print(len(impossible_list), len(set(impossible_list)))
# print the unique values in the impossible list and their counts in descending order
for office in sorted(set(impossible_list), key=impossible_list.count, reverse=True):
    count = impossible_list.count(office)
    print(f"{office}: {count}")

145672
52099 7005
Bayerische Kartei: 9925
Bundeskartei: 4884
HNG: 3402
Bayerisches Landesarchiv: 2585
Kreissozialamt: 958
Bundesamt: 834
KSHA: 750
Kreissozialhilfsausschuss: 712
Köln, Riehler Pl.2, 5000 Köln 1: 477
Doppelmeldung: 449
Bundesarchiv: 408
Erfurt: 386
An Bundeskartei: 371
BHF: 346
BEG: 340
Bayerisches Hauptstaatsarchiv: 321
Magistrat der Stadt Bremen: 306
Montabaur: 291
BMF: 269
K81: 268
K81n: 267
Frankfurt/M.: 267
Ansbach: 257
Sondervermögens-u. Bauverwaltung: 242
Bayerisches Landesamt: 241
Aurich/Ostfriesland: 235
Magistrat der Stadt Bremenhaven: 224
VB3: 218
Bayerisches Verfolgtenamt: 209
Landeszentrale für die Vergütung von Opfern des Nationalsozialismus: 206
KSH: 205
Köln, Riehler Pl. 2, 5000 Köln 1: 204
Watenstedt-Salzgitter: 185
Pirmasens: 181
Bayerisches Landeszentrum für die Opfer des Nationalsozialismus: 181
Kaiserslautern: 178
Bayerisches Landesamt für die Opfer des Nationalsozialismus: 173
Höherer Kommissar der Vereinten Nationen für Flüchtlinge: 166
Kreissonder

In [208]:
print(f"Max MatchDistance: {lev_matches['MatchDistance'].max()}")

Max MatchDistance: 58.0


In [209]:
total = len(lev_matches)-len(df_matches_impossible)
max_dist = int(lev_matches['MatchDistance'].max()) + 1
for i in range(1, max_dist):
    count = (lev_matches['MatchDistance'] == i).sum()
    pct = count / total * 100    
    print(f'MatchDistance = {i}: {count} ({pct:.2f}%)')

MatchDistance = 1: 17610 (12.09%)
MatchDistance = 2: 10216 (7.01%)
MatchDistance = 3: 9283 (6.37%)
MatchDistance = 4: 7249 (4.98%)
MatchDistance = 5: 25033 (17.18%)
MatchDistance = 6: 7157 (4.91%)
MatchDistance = 7: 5696 (3.91%)
MatchDistance = 8: 23344 (16.03%)
MatchDistance = 9: 5215 (3.58%)
MatchDistance = 10: 3719 (2.55%)
MatchDistance = 11: 5334 (3.66%)
MatchDistance = 12: 2539 (1.74%)
MatchDistance = 13: 3634 (2.49%)
MatchDistance = 14: 3022 (2.07%)
MatchDistance = 15: 2023 (1.39%)
MatchDistance = 16: 2193 (1.51%)
MatchDistance = 17: 2679 (1.84%)
MatchDistance = 18: 2706 (1.86%)
MatchDistance = 19: 1229 (0.84%)
MatchDistance = 20: 1014 (0.70%)
MatchDistance = 21: 1700 (1.17%)
MatchDistance = 22: 362 (0.25%)
MatchDistance = 23: 375 (0.26%)
MatchDistance = 24: 360 (0.25%)
MatchDistance = 25: 477 (0.33%)
MatchDistance = 26: 168 (0.12%)
MatchDistance = 27: 105 (0.07%)
MatchDistance = 28: 294 (0.20%)
MatchDistance = 29: 202 (0.14%)
MatchDistance = 30: 113 (0.08%)
MatchDistance = 31: 5

In [210]:
df_nomatches = lev_matches[(lev_matches['MatchDistance'] > 0) | (lev_matches['MatchDistance'].isna())]

df_nomatches.sort_values('MatchDistance', ascending=True).to_excel('compensation_office_nomatches.xlsx', index=False)

In [211]:
# build sets of all unique tokens in the ground truth and the extracted office names

def normalize(s):
    if not isinstance(s, str):
        return ""
    s = s.lower()
    replacements = {
        "ä": "ae", "ö": "oe", "ü": "ue", "ß": "ss"
    }
    for k, v in replacements.items():
        s = s.replace(k, v)
    s = unicodedata.normalize("NFKD", s)
    s = re.sub(r'[^a-zA-Z\s]', '', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

normalized_csv_token_gt = [normalize(office) for office in gt]
normalized_csv_token_corrections = [normalize(office) for office in list_corrections]
normalized_csv_token_further_corrections = [normalize(office) for office in list_further_corrections]
normalized_json_token_list = [normalize(office) for office in df_unmatched_filtered['CompensationOffice1'].tolist()]
normalized_csv_token_list = normalized_csv_token_gt + normalized_csv_token_corrections + normalized_csv_token_further_corrections

def office_token_set(series):
    tokens = set()
    for name in series:
        tokens.update(token for token in re.findall(r"\b\w+\b", name, flags=re.UNICODE)
            if not token.isdigit())
    return tokens

csv_tokens = office_token_set(normalized_csv_token_list)
json_tokens = office_token_set(normalized_json_token_list)

print(f"csv_tokens: {len(csv_tokens)}")
print(f"json_tokens: {len(json_tokens)}")

csv_tokens: 382
json_tokens: 10502


In [212]:
# map the tokens of the extracted office names to the closest equivalent in the ground truth tokens up to a maximal 
# edit distance of half of each token's length
json_to_csv_closest_map = {}
for json_token in json_tokens:
    if json_token in csv_tokens:
        continue
    
    max_distance = len(json_token) // 2
    best_match = None
    best_dist = float('inf')
    
    for csv_token in csv_tokens:
        if len(csv_token) >= 3:
            if abs(len(csv_token) - len(json_token)) > max_distance:
                continue
            dist = distance(json_token, csv_token)
            if dist <= max_distance and dist < best_dist:
                best_dist = dist
                best_match = csv_token
    
    if best_match is not None:
        json_to_csv_closest_map.setdefault(best_dist, []).append((json_token, best_match))

json_to_csv_closest_map = {
    k: sorted(v, key=lambda x: len(x[0]))
    for k, v in sorted(json_to_csv_closest_map.items())
}

print({k: len(v) for k, v in json_to_csv_closest_map.items()})
json_to_csv_closest_map

{1: 761, 2: 1133, 3: 1259, 4: 1047, 5: 821, 6: 634, 7: 386, 8: 310, 9: 222, 10: 139, 11: 76, 12: 38, 13: 23, 14: 12, 15: 11, 16: 5, 17: 3, 18: 1, 20: 3, 21: 1, 22: 1, 23: 1}


{1: [('rg', 'reg'),
  ('st', 'str'),
  ('ud', 'und'),
  ('re', 'reg'),
  ('ba', 'bad'),
  ('dr', 'der'),
  ('mn', 'min'),
  ('br', 'ibr'),
  ('et', 'etc'),
  ('da', 'das'),
  ('bd', 'bad'),
  ('de', 'die'),
  ('rb', 'lrb'),
  ('eg', 'reg'),
  ('ir', 'ibr'),
  ('di', 'die'),
  ('tr', 'str'),
  ('ne', 'neu'),
  ('es', 'des'),
  ('od', 'ofd'),
  ('lr', 'lrb'),
  ('fd', 'ofd'),
  ('as', 'das'),
  ('sr', 'str'),
  ('mi', 'min'),
  ('ib', 'ibr'),
  ('mt', 'amt'),
  ('of', 'ofd'),
  ('ges', 'des'),
  ('qfd', 'ofd'),
  ('mst', 'ist'),
  ('bes', 'des'),
  ('ung', 'und'),
  ('ksa', 'ksha'),
  ('lro', 'lrb'),
  ('cas', 'das'),
  ('win', 'fin'),
  ('dhe', 'die'),
  ('end', 'und'),
  ('ost', 'ist'),
  ('art', 'abt'),
  ('kst', 'ist'),
  ('efd', 'ofd'),
  ('sta', 'str'),
  ('rea', 'reg'),
  ('dep', 'dez'),
  ('stg', 'str'),
  ('lra', 'lrb'),
  ('ksh', 'ksha'),
  ('dem', 'dez'),
  ('fur', 'fuer'),
  ('beg', 'reg'),
  ('lfd', 'ofd'),
  ('ord', 'ofd'),
  ('mii', 'min'),
  ('ofr', 'ofd'),
  ('dfd', 'ofd

In [213]:
tokens_to_replace = []
tokens_to_replace.extend(json_to_csv_closest_map[1])
tokens_to_replace.extend(json_to_csv_closest_map[2])
tokens_to_replace.extend(json_to_csv_closest_map[3])
tokens_to_replace.remove(('main', 'mainz'))
tokens_to_replace.remove(('essen', 'hessen'))

In [214]:
unique_distance1_token_map = {
    json_token: csv_candidates
    for json_token, csv_candidates in tokens_to_replace
}

def replace_unique_distance1_tokens(text):
    text = "" if pd.isna(text) else str(text)
    return re.sub(
        r"\b\w+\b",
        lambda match: unique_distance1_token_map.get(match.group(0).lower(), match.group(0)),
        text,
        flags=re.UNICODE
    )

df_unmatched_filtered['CompensationOffice_token_swap'] = df_unmatched_filtered['CompensationOffice1'].apply(normalize).apply(replace_unique_distance1_tokens)

print(f"Unique token replacements available: {len(unique_distance1_token_map)}")

results = df_unmatched_filtered['CompensationOffice_token_swap'].apply(get_min_distance_dynamic_normalized)
df_matches_token_swap = pd.DataFrame({
    'CompensationOffice1': df_unmatched_filtered['CompensationOffice1'],
    'MatchedOffice': results.apply(lambda x: x[1]),
    'MatchDistance': results.apply(lambda x: x[0])
})

df_matches_token_swap.head(20)

Unique token replacements available: 3151


,CompensationOffice1,MatchedOffice,MatchDistance
7,Amt für Wiedergutmachung des Selbstkantkreises...,amtfuerwiedergutmachungdesselfkantkreisesgeile...,9.0
51,Kreisaußerhelfsausschübe,NaN,NaN
55,Der Innernister des Landes Nordrhein-Westfalen...,derinnenministerdeslandesnordrheinwestfalenabt...,6.0
72,Kreissonderhilfsschübe Peine,kreissonderhilfsausschussdelmenhorst,14.0
84,Regierungsbezirksamt für Wiedergutmachung und ...,regierungsbezirksamtfuerwiedergutmachungundver...,13.0
108,Bayerische Kartei,NaN,NaN
111,Entschädigungsamt,entschaedigungsamtberlin,6.0
113,Kreissonderhilfsausschuss Springe,kreissonderhilfsausschussdelmenhorst,10.0
115,Der Regierungspräsident - Dez. 14 - in Detmold,derregierungspraesidentindetmold,7.0
122,Bayerische Kartei,NaN,NaN


In [215]:
df_matches_token_swap[df_matches_token_swap['MatchDistance'] == 0]

,CompensationOffice1,MatchedOffice,MatchDistance
3519,Oldenburg/Old.,oldenburg,0.0
15458,Augsburg,arnsberg,0.0
20453,Begirussregierung,bezirksregierung,0.0
20946,Bekirskregierung,bezirksregierung,0.0
29889,Güldesheim,hildesheim,0.0
...,...,...,...
1885167,BfA-Berlin,berlin,0.0
1886447,Brauburg,freiburg,0.0
1888409,BFA-Berlin,berlin,0.0
1894003,Secerburg,saarburg,0.0


In [216]:
df_matches_token_swap[df_matches_token_swap['MatchDistance'].isin([1, 2])]

,CompensationOffice1,MatchedOffice,MatchDistance
1680,BayHues,bayern,1.0
3753,Regierung Arnswerg (Welf.),regierungarnsbergwestf,2.0
3920,Regierung Arnberg (Welf.),regierungarnsbergwestf,2.0
4507,Bayerisches Landesentschädigungsgesamt,bayerischeslandesentschaedigungsamt,1.0
7022,Regierungsamt Arnsherg (Welf.),regierungsamtarnsbergwestf,2.0
...,...,...,...
1895020,R.Pfz.-Berlin,rhpfalzberlin,2.0
1896680,Regierung Prensborg (Westf.),regierungarnsbergwestf,2.0
1896688,Bundesentschädigungssamt,blandesentschaedigungsamt,1.0
1897298,Regierung Finnsberg (Welf.),regierungarnsbergwestf,2.0


In [217]:
df_matches_token_swap[df_matches_token_swap['MatchDistance'].isin([1, 2])]

,CompensationOffice1,MatchedOffice,MatchDistance
1680,BayHues,bayern,1.0
3753,Regierung Arnswerg (Welf.),regierungarnsbergwestf,2.0
3920,Regierung Arnberg (Welf.),regierungarnsbergwestf,2.0
4507,Bayerisches Landesentschädigungsgesamt,bayerischeslandesentschaedigungsamt,1.0
7022,Regierungsamt Arnsherg (Welf.),regierungsamtarnsbergwestf,2.0
...,...,...,...
1895020,R.Pfz.-Berlin,rhpfalzberlin,2.0
1896680,Regierung Prensborg (Westf.),regierungarnsbergwestf,2.0
1896688,Bundesentschädigungssamt,blandesentschaedigungsamt,1.0
1897298,Regierung Finnsberg (Welf.),regierungarnsbergwestf,2.0
